# LOGISTPULSE · Deber 2 de Teoría de la Computación (USFQ)
Este notebook corre **todo el repositorio en Google Colab** sin instalar nada:
1. Ejecuta la celda **1** y sube el archivo `logistpulse-deber2-tc.zip` (o usa la celda 1-B si el repo ya está en GitHub).
2. Ejecuta las celdas en orden (`Entorno de ejecución → Ejecutar todas` también sirve).
3. En las celdas marcadas con ✏️ puedes cambiar los valores y volver a ejecutar para probar tus propios casos.

In [ ]:
# 1 · Subir y descomprimir el repo (Colab). Si ejecutas el notebook localmente desde la raíz del repo, no hace nada.
import os, sys, zipfile
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists("/content/logistpulse-deber2-tc"):
    from google.colab import files
    subido = files.upload()                      # elige logistpulse-deber2-tc.zip
    with zipfile.ZipFile(next(iter(subido))) as z:
        z.extractall("/content")
if EN_COLAB:
    os.chdir("/content/logistpulse-deber2-tc")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())
print("Contenido:", sorted(os.listdir(".")))

In [ ]:
# 1-B · ALTERNATIVA (repo público): borra el '#' de las dos líneas y pon tu usuario de GitHub.
# Al ser público, se clona sin iniciar sesión.
# !git clone https://github.com/USUARIO/logistpulse-deber2-tc.git /content/logistpulse-deber2-tc
# %cd /content/logistpulse-deber2-tc

## 2 · Pruebas automáticas (141: funcionales + complejidad)
Normales, límite e inválidas, más las pruebas que atan cada Big-O al código (conteo de operaciones y tiempo real). Al final debe decir **OK**.
Si Colab está muy cargado y falla solo una prueba de *tiempo*, vuelve a ejecutar; o desactívalas con `%env LOGISTPULSE_SIN_TIEMPOS=1`.

In [ ]:
!python -m unittest discover -s tests -v 2>&1 | tail -n 30

## 3 · Demo completa (autómatas, rutas, inventario y los 10 casos)

In [ ]:
!python -m logistpulse

## 4 · Diagramas (autómatas y planta con la ruta)

In [ ]:
!python scripts/generar_diagramas.py
from IPython.display import Image, display
for nombre in ["afd_pedido", "afd_reserva", "afd_orden_compra", "planta_ruta"]:
    print(nombre)
    display(Image(f"docs/diagramas/{nombre}.png", width=900))

## 5 · ✏️ Prueba tu propia secuencia en un autómata
Símbolos: pedido `c r i e d n x` · reserva `v s c p x t` · compra `s a r e p c f` (también puedes escribir los eventos por nombre).

In [ ]:
from logistpulse.automatas import PEDIDO, RESERVA, ORDEN_COMPRA

automata = PEDIDO          # ✏️ PEDIDO | RESERVA | ORDEN_COMPRA
secuencia = "criedn"       # ✏️ ej.: "crid", "x", ["confirmar", "reservar", "cancelar"]

r = automata.ejecutar(secuencia)
print(r.como_texto())

## 6 · ✏️ Ruta del AMR con zonas bloqueadas

In [ ]:
from logistpulse.io_datos import cargar_planta
from logistpulse.rutas import ruta_minima, todas_las_rutas_minimas

g, datos = cargar_planta()
origen, destino = "R", "D"   # ✏️ cualquier par de {R, A, B, C, P, E, D}
bloqueados = ["A"]           # ✏️ ej.: [], ["P"], ["E"]

ruta, costo = ruta_minima(g, origen, destino, bloqueados)
print("Ruta:", " → ".join(ruta) if ruta else "NO existe", "| costo:", costo, "m")
print("Todas las rutas de costo mínimo:", todas_las_rutas_minimas(g, origen, destino, bloqueados))

## 7 · ✏️ FEFO y reposición

In [ ]:
from logistpulse.io_datos import cargar_lotes
from logistpulse.inventario import seleccionar_fefo, EstadoSKU, sugerir

lotes, fecha_ref, _ = cargar_lotes()
r = seleccionar_fefo(lotes, "LECHE-1L", 120, fecha_ref, politica="parcial")   # ✏️ cantidad · politica: "parcial" | "rechazar" | "esperar"
print(r.estado, "| plan:", r.plan, "| pendiente:", r.pendiente)
print("Descartados:", r.descartados)

s = sugerir(EstadoSKU("SKU-EJ", disponible=18, en_orden=10, reservado=5, minimo=30, objetivo=50))  # ✏️
print(f"Posición = {s.posicion} → sugerencia = {s.cantidad}")

## 8 · ✏️ Verificar un certificado (grupo NP / NP-completo)
Buscar puede ser exponencial, pero **verificar** un certificado es polinomial.

In [ ]:
from logistpulse.io_datos import cargar_casos
from logistpulse.casos import c2_tsp_decision as c2, c3_vertex_cover as c3, n2_factor_rango as n2
casos = cargar_casos()                      # datos de los 10 expedientes (data/casos_simulador.json)

D, L = casos["c2"]["D"], casos["c2"]["L"]
print("TSP: ¿el tour (0,1,2,0) cuesta ≤ L?", c2.verificar(D, L, (0, 1, 2, 0)))            # ✏️ prueba otro tour
print("Vertex Cover: ¿{'B'} cubre con k=1?", c3.verificar(casos["c3"]["ady"], 1, {"B"}))   # ✏️ prueba {'A'}
print("Factor: ¿d=7 divide a N con 2 ≤ d ≤ B?", n2.verificar(casos["n2"]["N"], casos["n2"]["B"], 7))  # ✏️ prueba d=13


## 9 · Regenerar la documentación (tabla resumen, salidas observadas, tablas δ)
Las fichas ya existentes **no** se sobrescriben.

In [ ]:
!python scripts/generar_docs.py

## 10 · Tiempos reales: decidible ≠ resoluble en tiempo útil (Fase F y c2)
Mide la enumeración n!, Held–Karp, la heurística y Dijkstra en **esta** máquina. Cita estos números en el informe como "medidos en Colab".

In [ ]:
!python scripts/medir_crecimiento.py

## 11 · Revisión de la entrega contra la rúbrica
✅ listo · 🟡 pendiente (dice quién) · ❌ error. Córranlo antes de cada PR y antes de entregar.

In [ ]:
!python scripts/verificar_entrega.py

## 12 · Armar el informe (Word → PDF)
Junta los documentos de los tres en `docs/generado/INFORME.md` y `INFORME.docx`. Abre el .docx en Word/Google Docs, revisa, actualiza el índice y exporta a PDF como `docs/INFORME_Deber2_LOGISTPULSE.pdf`.

In [ ]:
import shutil
if shutil.which("pandoc") is None:
    !pip -q install pypandoc_binary
!python scripts/armar_informe.py
if EN_COLAB:
    from google.colab import files
    files.download("docs/generado/INFORME.docx")

## 13 · (Opcional) Descargar el repo con los cambios hechos en Colab

In [ ]:
if EN_COLAB:
    !cd /content && zip -qr logistpulse-actualizado.zip logistpulse-deber2-tc -x "*/__pycache__/*"
    from google.colab import files
    files.download("/content/logistpulse-actualizado.zip")
else:
    print("Solo aplica en Colab.")